In [ ]:
#| default_exp graph

## Notebook symbol graph

Statically match notebook definitions and calls so agents can see which cells define, call, and depend on a symbol.

This notebook adds a static map across notebooks. It does not execute project code; it parses cells with `ast`, records where symbols are defined, and reports which cells call those symbols.

That gives agents a fast way to answer questions like "where is this helper used?" before editing a private function or moving code between notebooks.

The graph is a navigation aid, not a runtime dependency analyzer. It answers practical maintenance questions: where is this symbol defined, who calls it, what does it call, can one symbol reach another through static calls, and are any notebooks importing private helpers that should stay local?

```python
symbol_graph(path="nbs", symbol="write_nb")
symbol_connection(path="nbs", start="symbol_connection", end="_resolve_callees")
private_symbol_report(path="nbs")
```

#### Production contract

The graph tools are maintenance aids, not runtime dependency analyzers. Production behavior is limited to static definitions, callers, callee summaries, order warnings, and private-boundary reports with modest claims about accuracy.

In [ ]:
#| export
import ast, builtins, copy, json, re
from pathlib import Path

from fastcore.basics import patch
from fastcore.nbio import read_nb
from nbskill.foundation import NotebookSymbol, call_name, cell_source, cli_error, cli_return, notebook_paths
from nbskill.foundation import parse_code_cell, path_candidates, source_without_directives, symbol_short_name
from nbskill.foundation import xml_attrs, xml_escape

In [ ]:
#| hide
from fastcore.test import test_eq

In [ ]:
#| export
_GRAPH_MAX_NOTEBOOKS = 80
_GRAPH_MAX_CELLS = 5000
_GRAPH_MAX_RECORDS = 20000
_GRAPH_MAX_EDGES = 40000
_GRAPH_MAX_SIMILARITY_RECORDS = 1000
_GRAPH_JSON_MAX_CHARS = 200000

In [ ]:
#| export
def _graph_limited_notebook_paths(path, max_notebooks=_GRAPH_MAX_NOTEBOOKS):
    paths = notebook_paths(path)
    if max_notebooks and len(paths) > max_notebooks:
        raise ValueError(f"Graph budget exceeded: {len(paths)} notebooks exceeds limit {max_notebooks}; narrow the path.")
    return paths

In [ ]:
#| export
def _graph_check_size(notebooks=0, cells=0, records=0, edges=0):
    if _GRAPH_MAX_NOTEBOOKS and notebooks > _GRAPH_MAX_NOTEBOOKS:
        raise ValueError(f"Graph budget exceeded: {notebooks} notebooks exceeds limit {_GRAPH_MAX_NOTEBOOKS}.")
    if _GRAPH_MAX_CELLS and cells > _GRAPH_MAX_CELLS:
        raise ValueError(f"Graph budget exceeded: {cells} cells exceeds limit {_GRAPH_MAX_CELLS}.")
    if _GRAPH_MAX_RECORDS and records > _GRAPH_MAX_RECORDS:
        raise ValueError(f"Graph budget exceeded: {records} records exceeds limit {_GRAPH_MAX_RECORDS}.")
    if _GRAPH_MAX_EDGES and edges > _GRAPH_MAX_EDGES:
        raise ValueError(f"Graph budget exceeded: {edges} edges exceeds limit {_GRAPH_MAX_EDGES}.")

## 1. Parse notebooks and collect evidence

The first chapter turns notebook cells into small, inspectable records. Each helper has one job so the order of the parser is visible to the reader.

In [ ]:
#| export
def _graph_scope(path):
    paths = notebook_paths(path)
    if paths:
        pth = Path(path)
        return pth.parent if pth.is_file() else pth
    pth = path_candidates(path)[-1]
    return pth.parent if pth.is_file() else pth

#### Discovering definitions and calls

The parser records function, class, and method definitions, then walks call expressions inside each parsed tree. Both fully qualified names and short names are kept because notebooks often import helpers into local scope.

In [ ]:
#| export
def _call_site_records(node, source=""):
    source_lines = str(source or "").splitlines()
    records = []
    for child in ast.walk(node):
        if not isinstance(child, ast.Call): continue
        name = call_name(child.func)
        if not name: continue
        lineno = getattr(child, "lineno", None)
        line = source_lines[lineno - 1].strip() if lineno and lineno <= len(source_lines) else ""
        for called_name in dict.fromkeys([name, name.rsplit(".", 1)[-1]]):
            records.append({"name": called_name, "lineno": lineno, "line": line})
    return tuple(records)

In [ ]:
#| export
def _call_names(node):
    return tuple(dict.fromkeys(site["name"] for site in _call_site_records(node)))

In [ ]:
#| export
def _node_definitions(node):
    if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)): return [(node.name, node, "function")]
    if isinstance(node, ast.ClassDef):
        items = [(node.name, node, "class")]
        for child in node.body:
            if isinstance(child, (ast.FunctionDef, ast.AsyncFunctionDef)):
                items.append((f"{node.name}.{child.name}", child, "method"))
        return items
    return []

#### Building graph records

Each code cell can contribute definition records and caller records. The collected graph is deliberately simple dictionaries so reports, tests, and future tools can inspect it without a graph database.

In [ ]:
#| export
def _cell_definition_records(path, module, idx, cell):
    tree = parse_code_cell(cell)
    if tree is None: return []
    records = []
    for node in tree.body:
        for symbol, symbol_node, kind in _node_definitions(node):
            records.append(dict(
                symbol=symbol, kind=kind, module=module, path=str(path),
                cell_id=getattr(cell, "id", ""), cell_idx=idx,
                calls=_call_names(symbol_node)
            ))
    return records

In [ ]:
#| export
def _cell_call_record(path, idx, cell):
    source = source_without_directives(cell_source(cell))
    tree = parse_code_cell(cell)
    if tree is None: return None
    calls = _call_names(tree)
    if not calls: return None
    return dict(
        path=str(path), cell_id=getattr(cell, "id", ""), cell_idx=idx,
        calls=calls, call_sites=_call_site_records(tree, source)
    )

In [ ]:
#| export
def _import_module_name(node):
    if node.module is None: return None
    if node.module == "nbskill": return ""
    if node.module.startswith("nbskill."): return node.module.removeprefix("nbskill.")
    return node.module

In [ ]:
#| export
def _cell_import_records(path, idx, cell):
    tree = parse_code_cell(cell)
    if tree is None: return []
    records = []
    for node in ast.walk(tree):
        if not isinstance(node, ast.ImportFrom): continue
        module = _import_module_name(node)
        if module is None: continue
        for alias in node.names:
            records.append(dict(
                module=module, symbol=alias.name, local=alias.asname or alias.name,
                path=str(path), cell_id=getattr(cell, "id", ""), cell_idx=idx
            ))
    return records

In [ ]:
#| export
def _notebook_module_name(path, nb):
    for cell in nb.cells:
        for line in cell_source(cell).splitlines():
            line = line.strip()
            if line.startswith("#| default_exp "):
                return line.split(None, 2)[-1].replace("/", ".")
    return Path(path).stem

In [ ]:
#| export
def _collect_graph(path="nbs"):
    definitions, callers, imports = [], [], []
    nb_paths = _graph_limited_notebook_paths(path)
    cell_count = 0
    for nb_path in nb_paths:
        nb = read_nb(nb_path)
        module = _notebook_module_name(nb_path, nb)
        for idx, cell in enumerate(nb.cells):
            cell_count += 1
            _graph_check_size(notebooks=len(nb_paths), cells=cell_count, records=len(definitions) + len(callers) + len(imports))
            definitions.extend(_cell_definition_records(nb_path, module, idx, cell))
            imports.extend(_cell_import_records(nb_path, idx, cell))
            record = _cell_call_record(nb_path, idx, cell)
            if record: callers.append(record)
            _graph_check_size(notebooks=len(nb_paths), cells=cell_count, records=len(definitions) + len(callers) + len(imports))
    return dict(definitions=definitions, callers=callers, imports=imports)

#### Cell order warnings

Notebook cells are executed in order, so a top-level call should not appear before the cell that defines or imports its callable. The order checker also looks inside function bodies for callable roots that are never defined or imported, which catches rarely exercised missing imports before runtime.

The checker treats names bound inside ordinary blocks as local to the cell. That matters for context-manager examples such as `with write_demo_notebook(...) as path:` and capture helpers such as `out = StringIO()`: those variables are setup, not missing imports.

In [ ]:
#| export
_BUILTIN_CALL_NAMES = set(dir(builtins)) | {"display", "get_ipython"}

In [ ]:
#| export
def _target_names(target):
    if isinstance(target, ast.Name): return {target.id}
    if isinstance(target, (ast.Tuple, ast.List)):
        names = set()
        for item in target.elts: names.update(_target_names(item))
        return names
    return set()

In [ ]:
#| export
def _argument_names(args):
    items = [
        *args.posonlyargs, *args.args, *args.kwonlyargs,
        *([args.vararg] if args.vararg else []),
        *([args.kwarg] if args.kwarg else []),
    ]
    return {arg.arg for arg in items if arg is not None}

In [ ]:
#| export
def _binding_names_from_node(node):
    if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)): return {node.name}
    if isinstance(node, ast.Import): return {alias.asname or alias.name.split(".", 1)[0] for alias in node.names}
    if isinstance(node, ast.ImportFrom):
        if any(alias.name == "*" for alias in node.names): return {"*"}
        return {alias.asname or alias.name for alias in node.names}
    if isinstance(node, ast.Assign):
        names = set()
        for target in node.targets: names.update(_target_names(target))
        return names
    if isinstance(node, ast.AnnAssign): return _target_names(node.target)
    if isinstance(node, (ast.For, ast.AsyncFor)): return _target_names(node.target)
    if isinstance(node, (ast.With, ast.AsyncWith)):
        names = set()
        for item in node.items:
            if item.optional_vars is not None: names.update(_target_names(item.optional_vars))
        return names
    if isinstance(node, ast.ExceptHandler) and node.name: return {node.name}
    return set()

In [ ]:
#| export
def _body_binding_names(nodes):
    names = set()
    for node in ast.walk(ast.Module(body=list(nodes), type_ignores=[])):
        names.update(_binding_names_from_node(node))
    return names

In [ ]:
#| export
def _block_binding_names(nodes):
    names = set()
    for node in nodes:
        names.update(_binding_names_from_node(node))
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.Lambda, ast.ClassDef)):
            continue
        for attr in ("body", "orelse", "finalbody"):
            body = getattr(node, attr, None)
            if body: names.update(_block_binding_names(body))
        for handler in getattr(node, "handlers", []):
            names.update(_binding_names_from_node(handler))
            names.update(_block_binding_names(handler.body))
    return names

In [ ]:
#| export
def _cell_binding_names(tree):
    return _block_binding_names(tree.body)

In [ ]:
#| export
def _call_root_name(node):
    if isinstance(node, ast.Name): return node.id
    if isinstance(node, ast.Attribute): return _call_root_name(node.value)
    return None

In [ ]:
#| export
class _CallRootVisitor(ast.NodeVisitor):
    def __init__(self):
        self.records = []
        self.local_scopes = []

In [ ]:
#| export
@patch
def _is_local(self: _CallRootVisitor, name):
    return any(name in scope for scope in self.local_scopes)

In [ ]:
#| export
@patch
def _with_scope(self: _CallRootVisitor, names, visit):
    self.local_scopes.append(names)
    visit()
    self.local_scopes.pop()

In [ ]:
#| export
@patch
def _visit_deferred_body(self: _CallRootVisitor, node, body):
    self._with_scope(_argument_names(node.args) | _body_binding_names(body), lambda: [self.visit(child) for child in body])

In [ ]:
#| export
@patch
def visit_FunctionDef(self: _CallRootVisitor, node):
    for item in [*node.decorator_list, *node.args.defaults, *node.args.kw_defaults]:
        if item is not None: self.visit(item)
    if node.returns is not None: self.visit(node.returns)
    self._visit_deferred_body(node, node.body)

In [ ]:
#| export
@patch
def visit_AsyncFunctionDef(self: _CallRootVisitor, node):
    self.visit_FunctionDef(node)

In [ ]:
#| export
@patch
def visit_Lambda(self: _CallRootVisitor, node):
    self._with_scope(_argument_names(node.args), lambda: self.visit(node.body))

In [ ]:
#| export
@patch
def _visit_comprehension(self: _CallRootVisitor, node):
    names = set()
    for generator in node.generators:
        names.update(_target_names(generator.target))
    def visit_body():
        for generator in node.generators:
            self.visit(generator.iter)
            for item in generator.ifs:
                self.visit(item)
        if hasattr(node, "elt"): self.visit(node.elt)
        if hasattr(node, "key"): self.visit(node.key)
        if hasattr(node, "value"): self.visit(node.value)
    self._with_scope(names, visit_body)

In [ ]:
#| export
@patch
def visit_ListComp(self: _CallRootVisitor, node):
    self._visit_comprehension(node)

In [ ]:
#| export
@patch
def visit_SetComp(self: _CallRootVisitor, node):
    self._visit_comprehension(node)

In [ ]:
#| export
@patch
def visit_DictComp(self: _CallRootVisitor, node):
    self._visit_comprehension(node)

In [ ]:
#| export
@patch
def visit_GeneratorExp(self: _CallRootVisitor, node):
    self._visit_comprehension(node)

In [ ]:
#| export
@patch
def visit_Call(self: _CallRootVisitor, node):
    name = _call_root_name(node.func)
    if name and not self._is_local(name):
        self.records.append(dict(symbol=name, line=getattr(node, "lineno", None), deferred=bool(self.local_scopes)))
    self.generic_visit(node)

In [ ]:
#| export
def _cell_call_root_records(cell):
    tree = parse_code_cell(cell)
    if tree is None: return []
    visitor = _CallRootVisitor()
    visitor.visit(tree)
    return visitor.records

In [ ]:
#| export
def _cell_order_data(path):
    data = []
    for nb_path in notebook_paths(path):
        try:
            nb = read_nb(nb_path)
        except FileNotFoundError:
            continue
        cells = []
        for idx, cell in enumerate(nb.cells):
            tree = parse_code_cell(cell)
            bindings = set() if tree is None else _cell_binding_names(tree)
            calls = _cell_call_root_records(cell)
            cells.append({"idx": idx, "cell": cell, "bindings": bindings, "calls": calls})
        data.append({"path": nb_path, "cells": cells})
    return data

In [ ]:
#| export
def _binding_locations(cells):
    locations = {}
    for item in cells:
        for name in item["bindings"]:
            locations.setdefault(name, []).append(item)
    return locations

In [ ]:
#| export
def _first_later_binding(locations, name, idx):
    return next((item for item in locations.get(name, []) if item["idx"] > idx), None)

In [ ]:
#| export
def _order_problem(kind, nb_path, item, call, detail, confidence="medium"):
    return dict(
        code=kind, path=str(nb_path), cell_id=getattr(item["cell"], "id", ""),
        line=call.get("line"), symbol=call["symbol"], detail=detail,
        severity="warning", source="nbskill", confidence=confidence
    )

In [ ]:
#| export
def _format_order_problem(problem):
    line = f" line={problem['line']}" if problem.get("line") else ""
    return f"- {problem['code']}: {problem['path']} id={problem.get('cell_id', '')}{line} symbol={problem['symbol']!r} {problem['detail']}"

In [ ]:
#| export
def notebook_order_problems(path="nbs"):
    "Return structured calls-before-definitions and missing callable import warnings."
    problems = []
    for nb_data in _cell_order_data(path):
        nb_path, cells = nb_data["path"], nb_data["cells"]
        locations = _binding_locations(cells)
        available = set(_BUILTIN_CALL_NAMES)
        for item in cells:
            cell_available = available | item["bindings"]
            star_imported = "*" in cell_available
            for call in item["calls"]:
                symbol = call["symbol"]
                if symbol in cell_available or star_imported: continue
                later = _first_later_binding(locations, symbol, item["idx"])
                if later and not call.get("deferred"):
                    problems.append(_order_problem(
                        "cell-order", nb_path, item, call,
                        f"called before definition/import in later cell id={getattr(later['cell'], 'id', '')}",
                        confidence="high",
                    ))
                elif not later:
                    problems.append(_order_problem(
                        "missing-import", nb_path, item, call,
                        "called without an earlier definition/import",
                        confidence="medium" if call.get("deferred") else "high",
                    ))
            available.update(item["bindings"])
    seen, unique = set(), []
    for problem in problems:
        key = tuple(problem.get(item) for item in ("code", "path", "cell_id", "line", "symbol"))
        if key in seen: continue
        seen.add(key)
        unique.append(problem)
    return unique

In [ ]:
#| export
def notebook_order_problem_lines(path="nbs"):
    "Return style-check lines for calls before definitions/imports and missing callable imports."
    return [_format_order_problem(problem) for problem in notebook_order_problems(path)]

## 2. Resolve and report relationships

The graph is approximate by design. Matching by exact or short symbol name is enough to surface likely callers and callees, which is the useful pre-edit signal for this project.

In [ ]:
#| export
def _call_matches_symbol(call, symbol):
    call = str(call)
    symbol = str(symbol)
    return call == symbol or symbol_short_name(call) == symbol_short_name(symbol)

In [ ]:
#| export
def _definitions_for_symbol(graph, symbol):
    return [record for record in graph["definitions"] if record["symbol"] == symbol or symbol_short_name(record["symbol"]) == symbol]

In [ ]:
#| export
def _caller_records_for_symbol(graph, symbol):
    return [record for record in graph["callers"] if any(_call_matches_symbol(call, symbol) for call in record["calls"])]

In [ ]:
#| export
def _resolve_callees(graph, calls):
    symbols = {record["symbol"] for record in graph["definitions"]}
    resolved = []
    for call in calls:
        for symbol in symbols:
            if _call_matches_symbol(call, symbol): resolved.append(symbol)
    return sorted(set(resolved))

In [ ]:
#| export
def _locations(records):
    return [f"{record['path']} id={record['cell_id']}" for record in records]

In [ ]:
#| export
def _callee_locations(graph, symbol):
    return _locations(_definitions_for_symbol(graph, symbol))

A symbol connection follows definition-to-callee edges with a bounded breadth-first search. That keeps the answer practical: show the shortest static call chain that explains how one symbol can reach another, and stop before approximate short-name matching turns into noise.

In [ ]:
#| export
def _definition_location(record):
    return dict(
        symbol=record["symbol"], kind=record.get("kind"), module=record.get("module"),
        path=record["path"], cell_id=record["cell_id"], cell_idx=record["cell_idx"]
    )

In [ ]:
#| export
def _symbol_start_nodes(graph, symbol):
    return sorted({record["symbol"] for record in _definitions_for_symbol(graph, symbol)})

In [ ]:
#| export
def _symbol_edge_records(graph, symbol):
    edges, seen = [], set()
    for record in _definitions_for_symbol(graph, symbol):
        for call in record.get("calls", ()):
            for callee in _resolve_callees(graph, [call]):
                if callee == record["symbol"]: continue
                key = (record["symbol"], callee, record["path"], record["cell_id"], call)
                if key in seen: continue
                seen.add(key)
                edge = dict(to=callee, call=call, from_location=_definition_location(record))
                edge["from"] = record["symbol"]
                edges.append(edge)
    return edges

In [ ]:
#| export
def _connection_symbols(start, chain):
    symbols = [chain[0]["from"]] if chain else [start]
    for edge in chain: symbols.append(edge["to"])
    return symbols

In [ ]:
#| export
def _symbol_connection_result(start, end, found, max_depth, symbols, chain, graph, reason=None):
    data = {
        "start": start,
        "end": end,
        "found": found,
        "max_depth": max_depth,
        "symbols": symbols,
        "chain": chain,
        "graph": graph,
    }
    if reason is not None: data["reason"] = reason
    return data

In [ ]:
#| export
def _symbol_connection_data(path, start, end, max_depth=6):
    graph = _collect_graph(_graph_scope(path))
    max_depth = max(0, int(max_depth))
    starts = _symbol_start_nodes(graph, start)
    if not starts:
        return _symbol_connection_result(start, end, False, max_depth, [], [], graph, reason="start symbol not found")
    queue = [(symbol, []) for symbol in starts]
    visited = set(starts)
    for symbol, chain in queue:
        if _call_matches_symbol(symbol, end):
            symbols = _connection_symbols(symbol, chain)
            return _symbol_connection_result(start, end, True, max_depth, symbols, chain, graph)
        if len(chain) >= max_depth: continue
        for edge in _symbol_edge_records(graph, symbol):
            callee = edge["to"]
            next_chain = [*chain, edge]
            if _call_matches_symbol(callee, end):
                symbols = _connection_symbols(symbol, next_chain)
                return _symbol_connection_result(start, end, True, max_depth, symbols, next_chain, graph)
            if callee in visited: continue
            visited.add(callee)
            queue.append((callee, next_chain))
    return _symbol_connection_result(start, end, False, max_depth, starts, [], graph, reason="no connection found")

#### Formatting reports

The reporting helpers turn graph records into compact text for humans and complete structured data for automation. Text output stays short enough for agent context, while JSON output keeps every caller usage line so migration scripts do not have to scrape truncated prose.

In [ ]:
#| export
def _caller_usage_records(records, symbol):
    items, seen = [], set()
    for record in records:
        for site in record.get("call_sites", ()):
            if not _call_matches_symbol(site.get("name"), symbol): continue
            key = (record["path"], record["cell_id"], site.get("lineno"), site.get("line"))
            if key in seen: continue
            seen.add(key)
            items.append(dict(
                path=record["path"], cell_id=record["cell_id"],
                lineno=site.get("lineno"), name=site.get("name"), line=site.get("line")
            ))
    return items

In [ ]:
#| export
def _caller_usage_lines(records, symbol, limit=8):
    lines = []
    for site in _caller_usage_records(records, symbol)[:limit]:
        lineno = f" line {site['lineno']}" if site.get("lineno") else ""
        source = f": {site['line']}" if site.get("line") else ""
        lines.append(f"- {site['path']} id={site['cell_id']}{lineno}{source}")
    return lines

In [ ]:
#| export
def symbol_graph_data(path, symbol):
    "Return structured definitions, callers, and callees for a symbol."
    graph = _collect_graph(_graph_scope(path))
    definitions = _definitions_for_symbol(graph, symbol)
    callers = _caller_records_for_symbol(graph, symbol)
    callee_symbols = []
    for definition in definitions:
        callee_symbols.extend(_resolve_callees(graph, definition["calls"]))
    callee_symbols = sorted(set(item for item in callee_symbols if item != symbol))
    return dict(
        symbol=symbol, definitions=definitions, callers=callers,
        caller_usages=_caller_usage_records(callers, symbol),
        callees=callee_symbols, graph=graph
    )

#### Symbols as graph entities

A symbol becomes more useful when its caller and callee relationships travel with it. The graph layer patches those relationship methods onto `NotebookSymbol` because this notebook owns the static call graph.

In [ ]:
#| export
@patch(cls_method=True, nm="from_graph_data")
def _from_graph_data(cls: NotebookSymbol, data):
    "Build a symbol model from `symbol_graph_data` output."
    definitions = data.get("definitions") or []
    record = definitions[0] if definitions else {"symbol": data.get("symbol", "")}
    return cls.from_record(record, data=data)

In [ ]:
#| export
@patch
def definitions(self: NotebookSymbol):
    "Return definition records for this symbol."
    return self.data.get("definitions", [])

In [ ]:
#| export
@patch
def callers(self: NotebookSymbol):
    "Return call records that reference this symbol."
    return self.data.get("callers", [])

In [ ]:
#| export
@patch
def caller_usages(self: NotebookSymbol):
    "Return formatted caller usage snippets for this symbol."
    return self.data.get("caller_usages", [])

In [ ]:
#| export
@patch
def callees(self: NotebookSymbol):
    "Return symbols called by this symbol."
    return self.data.get("callees", [])

In [ ]:
#| export
@patch
def relationships_to_record(self: NotebookSymbol):
    "Return this symbol with caller and callee relationship data."
    return {**self.to_record(), "definitions": self.definitions(), "callers": self.callers(), "callees": self.callees()}

In [ ]:
#| export
@patch
def relationships_to_xml(self: NotebookSymbol):
    "Return an XML-shaped symbol relationship view for LLM context."
    attrs = xml_attrs(symbol=self.name, path=self.path, cell_id=self.cell_id, cell_idx=self.cell_idx,
                       module=self.module, kind=self.kind)
    blocks = ["<definitions>"]
    blocks.extend(f"<location>{xml_escape(loc)}</location>" for loc in _locations(self.definitions()))
    blocks.append("</definitions>")
    blocks.append("<callers>")
    blocks.extend(f"<location>{xml_escape(loc)}</location>" for loc in _locations(self.callers()))
    blocks.extend(f"<usage>{xml_escape(item.get('line', ''))}</usage>" for item in self.caller_usages())
    blocks.append("</callers>")
    blocks.append("<callees>")
    blocks.extend(f"<symbol>{xml_escape(symbol)}</symbol>" for symbol in self.callees())
    blocks.append("</callees>")
    return f"<symbol_graph {attrs}>\n" + "\n".join(blocks) + "\n</symbol_graph>"

In [ ]:
_graph_example_data = {
    "symbol": "helper",
    "definitions": [{"symbol": "helper", "path": "demo.ipynb", "cell_id": "abc", "cell_idx": 1, "module": "demo", "kind": "function"}],
    "callers": [{"path": "demo.ipynb", "cell_id": "def", "cell_idx": 2, "calls": ["helper"]}],
    "caller_usages": [{"path": "demo.ipynb", "cell_id": "def", "lineno": 1, "line": "helper()"}],
    "callees": ["leaf"],
    "graph": {"definitions": []},
}
_graph_example = NotebookSymbol.from_graph_data(_graph_example_data)
{
    "definitions": _graph_example.definitions(),
    "callers": _graph_example.callers(),
    "caller_usages": _graph_example.caller_usages(),
    "callees": _graph_example.callees(),
}

In [ ]:
#| export
def symbol_graph_public_data(data):
    "Return the compact public view of symbol graph data."
    return dict(
        symbol=data["symbol"],
        definitions=data["definitions"][:200],
        callers=data["callers"][:200],
        caller_usages=data["caller_usages"][:50],
        callees=[
            dict(symbol=symbol, locations=_callee_locations(data["graph"], symbol))
            for symbol in data["callees"][:200]
        ],
    )

In [ ]:
#| export
def _format_symbol_graph_data(data):
    model = NotebookSymbol.from_graph_data(data)
    lines = [f"Symbol {model.name}"]
    lines.append("Definitions:")
    lines.extend([f"- {loc}" for loc in _locations(model.definitions())] or ["- (none)"])
    lines.append("Callers:")
    lines.extend([f"- {loc}" for loc in _locations(model.callers())] or ["- (none)"])
    caller_usage = _caller_usage_lines(model.callers(), model.name)
    if caller_usage:
        lines.append("Caller usages:")
        lines.extend(caller_usage)
    lines.append("Callees:")
    if model.callees():
        for symbol in model.callees():
            locs = "; ".join(_callee_locations(data["graph"], symbol)) or "unknown location"
            lines.append(f"- {symbol}: {locs}")
    else:
        lines.append("- (none)")
    return "\n".join(lines)

In [ ]:
#| hide
_graph_model_data = {
    "symbol": "helper",
    "definitions": [{"symbol": "helper", "path": "demo.ipynb", "cell_id": "abc", "cell_idx": 1, "module": "demo", "kind": "function"}],
    "callers": [{"path": "demo.ipynb", "cell_id": "def", "cell_idx": 2, "calls": ["helper"]}],
    "caller_usages": [{"path": "demo.ipynb", "cell_id": "def", "lineno": 1, "line": "helper()"}],
    "callees": ["leaf"],
    "graph": {"definitions": []},
}
_graph_model = NotebookSymbol.from_graph_data(_graph_model_data)
assert _graph_model.name == "helper"
assert _graph_model.definitions()[0]["cell_id"] == "abc"
assert _graph_model.callers()[0]["cell_id"] == "def"
assert _graph_model.caller_usages()[0]["line"] == "helper()"
assert _graph_model.callees() == ["leaf"]
assert _graph_model.relationships_to_record()["callees"] == ["leaf"]
assert "helper()" in _graph_model.relationships_to_xml()
assert _format_symbol_graph_data(_graph_model_data).startswith("Symbol helper")

In [ ]:
#| export
def symbol_usage_summary(path, symbols):
    "Return a compact caller/callee summary for one or more symbols."
    if isinstance(symbols, str): symbols = [symbols]
    chunks = []
    for symbol in dict.fromkeys(symbols or []):
        data = symbol_graph_data(path, symbol)
        callers = "; ".join(_locations(data["callers"])[:8]) or "none"
        callees = "; ".join(data["callees"][:8]) or "none"
        chunks.append(f"{symbol}: callers={callers}; callees={callees}")
        caller_usage = _caller_usage_lines(data["callers"], symbol)
        if caller_usage:
            chunks.append("Caller usages:")
            chunks.extend(caller_usage)
    return "\n".join(chunks)

In [ ]:
#| export
def _symbol_connection_public_data(data):
    return {
        key: data[key]
        for key in ("start", "end", "found", "max_depth", "symbols", "chain", "reason")
        if key in data
    }

In [ ]:
#| export
def _format_symbol_connection_data(data):
    lines = [f"Symbol connection {data['start']} -> {data['end']}"]
    if not data["found"]:
        reason = data.get("reason", "no connection found")
        lines.append(f"No connection found within {data['max_depth']} edge(s): {reason}.")
        if data["symbols"]: lines.append("Matched starts: " + ", ".join(data["symbols"]))
        return "\n".join(lines)
    lines.append("Symbols: " + " -> ".join(data["symbols"]))
    lines.append("Chain:")
    for edge in data["chain"]:
        loc = edge["from_location"]
        lines.append(
            f"- {edge['from']} -> {edge['to']}: "
            f"{loc['path']} id={loc['cell_id']} calls {edge['call']}"
        )
    if not data["chain"]: lines.append("- start and end matched the same symbol")
    return "\n".join(lines)

#### Public graph reports

`symbol_graph` focuses on one symbol's definitions, callers, caller usage lines, and callees. `symbol_connection` answers the next question: whether one symbol can reach another by following static callee edges, returning the shortest chain it finds. Default output is compact text for quick orientation; `json_output=True` returns structured data for programmatic refactors. `private_symbol_report` looks for cross-notebook calls to private helpers, which is useful when deciding whether a private function can be changed safely.

In [ ]:
#| export
def symbol_graph(
    path: str = "nbs",  # Notebook file, directory, or glob to scan
    symbol: str = "",  # Function, class, or Class.method to inspect
    json_output: bool = False,  # Print complete structured JSON instead of compact text
):
    "Print definitions, callers, and callees for a notebook symbol."
    if not symbol: cli_error("Pass --symbol to inspect")
    data = symbol_graph_data(path, symbol)
    if json_output:
        result = symbol_graph_public_data(data)
        text = json.dumps(result, indent=2, sort_keys=True)
        print(text)
        return cli_return(result)
    text = _format_symbol_graph_data(data)
    print(text)
    return cli_return(text)

In [ ]:
#| export
def private_symbol_report(
    path: str = "nbs",  # Notebook file, directory, or glob to scan
):
    "Print cross-notebook calls to imported private `_` symbols."
    graph = _collect_graph(path)
    definitions = {(record.get("module"), record["symbol"]): record for record in graph["definitions"]}
    lines = ["Cross-notebook private symbol calls"]
    for imported in graph["imports"]:
        symbol = imported["symbol"]
        if not symbol_short_name(symbol).startswith("_"): continue
        definition = definitions.get((imported["module"], symbol))
        if not definition or definition["path"] == imported["path"]: continue
        for caller in graph["callers"]:
            if caller["path"] != imported["path"]: continue
            if not any(_call_matches_symbol(call, imported["local"]) for call in caller["calls"]): continue
            lines.append(
                f"- {symbol} defined {definition['path']} id={definition['cell_id']} "
                f"called from {caller['path']} id={caller['cell_id']}"
            )
    if len(lines) == 1: lines.append("No cross-notebook private symbol calls found.")
    text = "\n".join(dict.fromkeys(lines))
    print(text)
    return cli_return(text)

In [ ]:
#| export
def symbol_connection(
    path: str = "nbs",  # Notebook file, directory, or glob to scan
    start: str = "",  # Symbol where traversal starts
    end: str = "",  # Symbol to find by following callees
    max_depth: int = 6,  # Maximum number of edges to follow
    json_output: bool = False,  # Print complete structured JSON instead of compact text
):
    "Print the shortest static callee chain connecting two notebook symbols."
    if not start or not end: cli_error("Pass --start and --end to connect symbols")
    data = _symbol_connection_data(path, start, end, max_depth=max_depth)
    if json_output:
        result = _symbol_connection_public_data(data)
        text = json.dumps(result, indent=2, sort_keys=True)
        print(text)
        return cli_return(result)
    text = _format_symbol_connection_data(data)
    print(text)
    return cli_return(text)

In [ ]:
#| hide
from contextlib import redirect_stdout
from io import StringIO
from fastcore.nbio import mk_cell, new_nb, write_nb
from nbskill.graph import symbol_graph as _example_symbol_graph
from nbskill.foundation import demo_path, remove_demo_path, write_demo_notebook
from nbskill.mcp import capture_call
from nbskill.graph import private_symbol_report, symbol_graph, symbol_usage_summary

In [ ]:
root = demo_path("10_graph_example")
try:
    root.mkdir()
    write_nb(new_nb([
        mk_cell("#| default_exp demo"),
        mk_cell("#| export\ndef helper():\n    return 1\n\ndef target():\n    return helper()"),
        mk_cell("target()"),
    ]), root / "demo.ipynb")
    _example_symbol_graph(str(root), "helper")
finally:
    remove_demo_path(root)

In [ ]:
#| export
class _ShapeNormalizer(ast.NodeTransformer):
    def visit_FunctionDef(self, node):
        node.name = "_"
        return self.generic_visit(node)

    def visit_AsyncFunctionDef(self, node):
        node.name = "_"
        return self.generic_visit(node)

    def visit_ClassDef(self, node):
        node.name = "_"
        return self.generic_visit(node)

    def visit_arg(self, node):
        node.arg = "_"
        return self.generic_visit(node)

    def visit_Name(self, node):
        node.id = "_"
        return node

    def visit_Constant(self, node):
        node.value = "_"
        return node

In [ ]:
#| export
def _tokens_from_text(text):
    return sorted(set(re.findall(r"[a-z0-9]+", str(text).lower())))

In [ ]:
#| export
def _normalized_ast_shape(node):
    normalized = copy.deepcopy(node)
    normalized = _ShapeNormalizer().visit(normalized)
    ast.fix_missing_locations(normalized)
    return ast.dump(normalized, annotate_fields=False, include_attributes=False)

In [ ]:
#| export
def _decorator_names(node):
    return sorted(filter(None, (call_name(item) for item in getattr(node, "decorator_list", []))))

In [ ]:
#| export
def _cell_import_names(tree):
    names = []
    for node in ast.walk(tree):
        if isinstance(node, ast.Import):
            names.extend(alias.name for alias in node.names)
        elif isinstance(node, ast.ImportFrom):
            module = node.module or ""
            for alias in node.names:
                names.append(f"{module}.{alias.name}" if module else alias.name)
    return sorted(set(names))

In [ ]:
#| export
def _markdown_heading(cell):
    if getattr(cell, "cell_type", "") != "markdown": return ""
    lines = str(cell_source(cell)).splitlines()
    headings = [line.strip("# ").strip() for line in lines if line.lstrip().startswith("#")]
    return " ".join(headings)

In [ ]:
#| export
def _heading_before(cells, idx):
    for cell in reversed(cells[:idx]):
        heading = _markdown_heading(cell)
        if heading: return heading
    return ""

In [ ]:
#| export
def _notebook_text_profile(path, nb, module):
    headings = [_markdown_heading(cell) for cell in nb.cells]
    headings = [heading for heading in headings if heading]
    imports = []
    symbols = []
    for idx, cell in enumerate(nb.cells):
        tree = parse_code_cell(cell)
        if tree is None: continue
        imports.extend(_cell_import_names(tree))
        for node in tree.body:
            symbols.extend(symbol for symbol, _, _ in _node_definitions(node))
    text = " ".join([module, Path(path).stem, *headings, *symbols, *imports])
    return dict(
        path=str(path), module=module, headings=headings,
        symbols=sorted(set(symbols)), imports=sorted(set(imports)),
        tokens=_tokens_from_text(text)
    )

In [ ]:
#| export
def _catalog_record(path, module, idx, cell, node, symbol, kind, heading, imports):
    docstring = ast.get_docstring(node) or ""
    text = " ".join([symbol, kind, module, heading, docstring, " ".join(imports)])
    exported = str(cell_source(cell)).lstrip().startswith("#| export")
    return dict(
        symbol=symbol, kind=kind, module=module, path=str(path),
        cell_id=getattr(cell, "id", ""), cell_idx=idx, exported=exported,
        private=symbol_short_name(symbol).startswith("_"), docstring=docstring,
        heading=heading, tokens=_tokens_from_text(text),
        calls=sorted(set(_call_names(node))), imports=imports,
        decorators=_decorator_names(node), ast_shape=_normalized_ast_shape(node)
    )

In [ ]:
#| export
def symbol_catalog(path='nbs'):
    """Return notebook symbols with placement and reuse evidence."""
    symbols = []
    notebooks = []
    nb_paths = _graph_limited_notebook_paths(path)
    cell_count = 0
    for nb_path in nb_paths:
        nb = read_nb(nb_path)
        module = _notebook_module_name(nb_path, nb)
        notebooks.append(_notebook_text_profile(nb_path, nb, module))
        for idx, cell in enumerate(nb.cells):
            cell_count += 1
            _graph_check_size(notebooks=len(nb_paths), cells=cell_count, records=len(symbols))
            tree = parse_code_cell(cell)
            if tree is None:
                continue
            heading = _heading_before(nb.cells, idx)
            imports = _cell_import_names(tree)
            for node in tree.body:
                for symbol, symbol_node, kind in _node_definitions(node):
                    symbols.append(
                        _catalog_record(
                            nb_path,
                            module,
                            idx,
                            cell,
                            symbol_node,
                            symbol,
                            kind,
                            heading,
                            imports,
                        )
                    )
                    _graph_check_size(
                        notebooks=len(nb_paths),
                        cells=cell_count,
                        records=len(symbols),
                    )
    return dict(
        symbols=symbols,
        notebooks=notebooks,
        graph=_collect_graph(path),
    )

In [ ]:
#| hide
test_eq(isinstance(symbol_catalog("nbs/10_graph.ipynb"), dict), True)
test_eq({"symbols", "notebooks", "graph"} <= set(symbol_catalog("nbs/10_graph.ipynb")), True)

## 3. Build a notebook knowledge graph

`notebook_knowledge_graph_data` exports a typed, deterministic graph for local notebook structure. Edges carry confidence and evidence so call/import facts stay separate from heuristic similarity links.

In [ ]:
#| export
_KG_NODE_TYPES = {"notebook", "cell", "symbol", "import", "heading"}
_KG_EDGE_TYPES = {"contains", "defines", "calls", "imports", "documents", "precedes", "similar_to"}
_KG_REQUIRED_TOP_KEYS = {"version", "kind", "project", "nodes", "edges", "layers", "tour", "issues"}
_KG_REQUIRED_NODE_KEYS = {"id", "type", "name"}
_KG_REQUIRED_EDGE_KEYS = {"source", "target", "type", "confidence", "evidence"}

In [ ]:
#| export
def _kg_id(*parts):
    return "::".join(str(part).replace("::", ":") for part in parts if part not in (None, ""))

In [ ]:
#| export
def _kg_clean(extra):
    return {key: value for key, value in extra.items() if value is not None and value != ""}

In [ ]:
#| export
def _kg_project_summary(scope, ordered_paths, state):
    return dict(
        path=str(scope), notebook_count=len(ordered_paths),
        node_count=len(state["nodes"]), edge_count=len(state["edges"])
    )

In [ ]:
#| export
def _kg_node(node_id, kind, name, **extra):
    data = dict(id=node_id, type=kind, name=str(name))
    data.update(_kg_clean(extra))
    return data

In [ ]:
#| export
def _kg_edge(source, target, kind, confidence="high", evidence=""):
    return dict(source=source, target=target, type=kind, confidence=confidence, evidence=evidence)

In [ ]:
#| export
def _kg_issue(code, detail, severity="warning", **extra):
    data = dict(code=code, detail=detail, severity=severity)
    data.update(_kg_clean(extra))
    return data

In [ ]:
#| export
def _notebook_node_id(path):
    return _kg_id("notebook", path)

In [ ]:
#| export
def _cell_node_id(path, cell_idx, cell_id=None):
    return _kg_id("cell", path, cell_id or cell_idx)

In [ ]:
#| export
def _symbol_node_id(record):
    return _kg_id("symbol", record.get("module"), record.get("symbol"), record.get("cell_id"))

In [ ]:
#| export
def _import_node_id(record):
    return _kg_id("import", record.get("path"), record.get("cell_id"), record.get("local") or record.get("symbol"))

In [ ]:
#| export
def _heading_node_id(path, cell_idx, title):
    return _kg_id("heading", path, cell_idx, title)

In [ ]:
#| export
def _add_kg_node(nodes, node):
    nodes.setdefault(node["id"], node)
    return node["id"]

In [ ]:
#| export
def _add_kg_edge(edges, seen, edge):
    if _GRAPH_MAX_EDGES and len(edges) >= _GRAPH_MAX_EDGES: return
    key = (edge["source"], edge["target"], edge["type"], edge.get("evidence", ""))
    if key not in seen:
        edges.append(edge)
        seen.add(key)

In [ ]:
#| export
def _append_layer_node(layers, layer_id, name, node_id, kind="notebook"):
    layer = layers.setdefault(layer_id, {"id": layer_id, "type": kind, "name": name, "node_ids": []})
    if node_id not in layer["node_ids"]:
        layer["node_ids"].append(node_id)
    return layer

In [ ]:
#| export
def _kg_records_by_cell(records):
    by_cell = {}
    for record in records:
        by_cell.setdefault((record.get("path"), record.get("cell_id")), []).append(record)
    return by_cell

In [ ]:
#| export
def _symbol_ids_by_name(nodes):
    by_name = {}
    for node in nodes.values():
        if node.get("type") == "symbol":
            by_name.setdefault(node.get("name"), []).append(node["id"])
    return by_name

In [ ]:
#| export
def _validate_kg_top_level(data):
    return [
        _kg_issue("missing-top-level-field", f"Missing top-level field: {key}.", field=key)
        for key in sorted(_KG_REQUIRED_TOP_KEYS - set(data))
    ]

In [ ]:
#| export
def _validate_kg_nodes(nodes):
    issues, node_ids, duplicate_ids = [], set(), set()
    for idx, node in enumerate(nodes):
        missing = _KG_REQUIRED_NODE_KEYS - set(node)
        if missing:
            issues.append(_kg_issue("missing-node-field", f"Node {idx} is missing: {', '.join(sorted(missing))}.", node_index=idx))
        node_id = node.get("id")
        if node_id in node_ids:
            duplicate_ids.add(node_id)
        if node_id:
            node_ids.add(node_id)
        if node.get("type") and node.get("type") not in _KG_NODE_TYPES:
            issues.append(_kg_issue("unknown-node-type", f"Unknown node type: {node.get('type')}.", node_id=node_id))
    for node_id in sorted(duplicate_ids):
        issues.append(_kg_issue("duplicate-node-id", f"Duplicate node id: {node_id}.", node_id=node_id))
    return issues, node_ids

In [ ]:
#| export
def _validate_kg_edges(edges, node_ids):
    issues = []
    for idx, edge in enumerate(edges):
        missing = _KG_REQUIRED_EDGE_KEYS - set(edge)
        if missing:
            issues.append(_kg_issue("missing-edge-field", f"Edge {idx} is missing: {', '.join(sorted(missing))}.", edge_index=idx))
            continue
        if edge.get("type") not in _KG_EDGE_TYPES:
            issues.append(_kg_issue("unknown-edge-type", f"Unknown edge type: {edge.get('type')}.", edge_index=idx))
        for field in ("source", "target"):
            ref = edge.get(field)
            if ref not in node_ids:
                issues.append(_kg_issue("dangling-edge", f"Edge {idx} has unknown {field}: {ref}.", edge_index=idx, field=field, node_id=ref))
    return issues

In [ ]:
#| export
def _validate_kg_node_refs(items, node_ids, code, label):
    issues = []
    for item in items:
        for node_id in item.get("node_ids", []):
            if node_id not in node_ids:
                detail = f"{label} {item.get('id', item.get('order'))} references unknown node: {node_id}."
                issues.append(_kg_issue(code, detail, item_id=item.get("id"), order=item.get("order"), node_id=node_id))
    return issues

In [ ]:
#| export
def _notebook_knowledge_graph_validate(data):
    if not isinstance(data, dict):
        return [_kg_issue("invalid-graph", "Graph data must be a dictionary.", severity="error")]
    node_issues, node_ids = _validate_kg_nodes(data.get("nodes", []))
    issues = _validate_kg_top_level(data) + node_issues
    issues.extend(_validate_kg_edges(data.get("edges", []), node_ids))
    issues.extend(_validate_kg_node_refs(data.get("layers", []), node_ids, "dangling-layer-node", "Layer"))
    issues.extend(_validate_kg_node_refs(data.get("tour", []), node_ids, "dangling-tour-node", "Tour item"))
    return issues

In [ ]:
#| export
def _kg_state():
    return {"nodes": {}, "edges": [], "edge_seen": set(), "layers": {}, "tour": [], "symbol_nodes_by_name": {}}

In [ ]:
#| export
def _kg_cell_base(state, nb_path, module, notebook_id, layer_id, cell_idx, cell, previous_cell_id, heading_layer):
    cell_id = getattr(cell, "id", "")
    node_id = _add_kg_node(state["nodes"], _kg_node(_cell_node_id(nb_path, cell_idx, cell_id), "cell", f"{Path(nb_path).name}#{cell_idx}", path=nb_path, cell_id=cell_id, cell_idx=cell_idx, cell_type=cell.cell_type))
    _append_layer_node(state["layers"], layer_id, module, node_id, kind="module")
    if heading_layer:
        _append_layer_node(state["layers"], heading_layer["id"], heading_layer["name"], node_id, kind="heading")
    _add_kg_edge(state["edges"], state["edge_seen"], _kg_edge(notebook_id, node_id, "contains", evidence=f"cell {cell_idx}"))
    if previous_cell_id:
        _add_kg_edge(state["edges"], state["edge_seen"], _kg_edge(previous_cell_id, node_id, "precedes", evidence="notebook cell order"))
    return node_id, cell_id

In [ ]:
#| export
def _kg_heading(state, nb_path, notebook_id, cell_idx, cell, cell_id, node_id, heading_layer):
    heading = _markdown_heading(cell)
    if not heading:
        return heading_layer, None
    heading_id = _add_kg_node(state["nodes"], _kg_node(_heading_node_id(nb_path, cell_idx, heading), "heading", heading, path=nb_path, cell_id=cell_id, cell_idx=cell_idx))
    _add_kg_edge(state["edges"], state["edge_seen"], _kg_edge(notebook_id, heading_id, "contains", evidence=f"heading at cell {cell_idx}"))
    _add_kg_edge(state["edges"], state["edge_seen"], _kg_edge(heading_id, node_id, "documents", evidence="markdown heading cell"))
    layer = _append_layer_node(state["layers"], _kg_id("heading", nb_path, cell_idx), heading, heading_id, kind="heading")
    _append_layer_node(state["layers"], layer["id"], layer["name"], node_id, kind="heading")
    return layer, heading_id

In [ ]:
#| export
def _kg_definition_nodes(state, nb_path, module, layer_id, cell_idx, cell_id, node_id, records, heading_layer):
    for record in records:
        symbol_id = _add_kg_node(state["nodes"], _kg_node(_symbol_node_id(record), "symbol", record["symbol"], symbol_kind=record.get("kind"), module=record.get("module"), path=nb_path, cell_id=cell_id, cell_idx=cell_idx, private=record.get("symbol", "").startswith("_")))
        state["symbol_nodes_by_name"].setdefault(record["symbol"], []).append(symbol_id)
        _append_layer_node(state["layers"], layer_id, module, symbol_id, kind="module")
        if heading_layer:
            _append_layer_node(state["layers"], heading_layer["id"], heading_layer["name"], symbol_id, kind="heading")
        _add_kg_edge(state["edges"], state["edge_seen"], _kg_edge(node_id, symbol_id, "defines", evidence=f"definition in cell {cell_idx}"))

In [ ]:
#| export
def _kg_import_nodes(state, nb_path, module, layer_id, cell_idx, cell_id, node_id, records, heading_layer):
    for record in records:
        label = record.get("local") or record.get("symbol") or record.get("module")
        import_id = _add_kg_node(state["nodes"], _kg_node(_import_node_id(record), "import", label, module=record.get("module"), symbol=record.get("symbol"), local=record.get("local"), path=nb_path, cell_id=cell_id, cell_idx=cell_idx))
        _append_layer_node(state["layers"], layer_id, module, import_id, kind="module")
        if heading_layer:
            _append_layer_node(state["layers"], heading_layer["id"], heading_layer["name"], import_id, kind="heading")
        _add_kg_edge(state["edges"], state["edge_seen"], _kg_edge(node_id, import_id, "imports", evidence=f"import in cell {cell_idx}"))

In [ ]:
#| export
def _kg_add_notebook(state, nb_path, nb_order, graph):
    nb = read_nb(nb_path)
    module = _notebook_module_name(nb_path, nb)
    notebook_id = _add_kg_node(state["nodes"], _kg_node(_notebook_node_id(nb_path), "notebook", Path(nb_path).name, path=nb_path, module=module, cell_count=len(nb.cells)))
    layer_id = _kg_id("module", module)
    _append_layer_node(state["layers"], layer_id, module, notebook_id, kind="module")
    definitions = _kg_records_by_cell(graph["definitions"])
    imports = _kg_records_by_cell(graph["imports"])
    heading_layer, previous_cell_id, first_heading_id = None, None, None
    for cell_idx, cell in enumerate(nb.cells):
        node_id, cell_id = _kg_cell_base(state, nb_path, module, notebook_id, layer_id, cell_idx, cell, previous_cell_id, heading_layer)
        previous_cell_id = node_id
        heading_layer, heading_id = _kg_heading(state, nb_path, notebook_id, cell_idx, cell, cell_id, node_id, heading_layer)
        first_heading_id = first_heading_id or heading_id
        records_key = (nb_path, cell_id)
        _kg_definition_nodes(state, nb_path, module, layer_id, cell_idx, cell_id, node_id, definitions.get(records_key, []), heading_layer)
        _kg_import_nodes(state, nb_path, module, layer_id, cell_idx, cell_id, node_id, imports.get(records_key, []), heading_layer)
    tour_nodes = [notebook_id]
    if first_heading_id:
        tour_nodes.append(first_heading_id)
    state["tour"].append({"order": nb_order, "title": module, "node_ids": tour_nodes, "description": f"{module}: {Path(nb_path).name}"})

In [ ]:
#| export
def _merge_symbol_nodes_by_name(state):
    for name, ids in _symbol_ids_by_name(state["nodes"]).items():
        bucket = state["symbol_nodes_by_name"].setdefault(name, [])
        for node_id in ids:
            if node_id not in bucket:
                bucket.append(node_id)

In [ ]:
#| export
def _kg_add_call_edges(state, graph):
    _merge_symbol_nodes_by_name(state)
    for record in graph["definitions"]:
        if _GRAPH_MAX_EDGES and len(state["edges"]) >= _GRAPH_MAX_EDGES: break
        source_id = _symbol_node_id(record)
        for call in record.get("calls", []):
            for callee in _resolve_callees(graph, [call]):
                for target_id in state["symbol_nodes_by_name"].get(callee, []):
                    if source_id != target_id:
                        _add_kg_edge(state["edges"], state["edge_seen"], _kg_edge(source_id, target_id, "calls", confidence="high", evidence=f"{record['symbol']} calls {call}"))
    for record in graph["callers"]:
        if _GRAPH_MAX_EDGES and len(state["edges"]) >= _GRAPH_MAX_EDGES: break
        source_id = _cell_node_id(record.get("path"), record.get("cell_idx"), record.get("cell_id"))
        for call in record.get("calls", []):
            for callee in _resolve_callees(graph, [call]):
                for target_id in state["symbol_nodes_by_name"].get(callee, []):
                    _add_kg_edge(state["edges"], state["edge_seen"], _kg_edge(source_id, target_id, "calls", confidence="high", evidence=f"cell calls {call}"))

In [ ]:
#| export
def _add_similarity_edges(catalog, nodes, edges, seen):
    symbols = catalog.get("symbols", catalog) if isinstance(catalog, dict) else catalog
    records = [record for record in symbols if record.get("ast_shape")][:_GRAPH_MAX_SIMILARITY_RECORDS]
    ids_by_key = {(node.get("module"), node.get("name")): node["id"] for node in nodes.values() if node.get("type") == "symbol"}
    for idx, left in enumerate(records):
        if _GRAPH_MAX_EDGES and len(edges) >= _GRAPH_MAX_EDGES: break
        for right in records[idx + 1:]:
            if _GRAPH_MAX_EDGES and len(edges) >= _GRAPH_MAX_EDGES: break
            if left.get("symbol") == right.get("symbol"):
                continue
            if left.get("ast_shape") != right.get("ast_shape"):
                continue
            source = ids_by_key.get((left.get("module"), left.get("symbol")))
            target = ids_by_key.get((right.get("module"), right.get("symbol")))
            if source and target:
                _add_kg_edge(edges, seen, _kg_edge(source, target, "similar_to", confidence="medium", evidence="normalized AST shape matches"))

In [ ]:
#| export
def _format_notebook_knowledge_graph(data):
    project = data.get("project", {})
    lines = [
        f"Notebook knowledge graph: {project.get('notebook_count', 0)} notebook(s), {len(data.get('nodes', []))} node(s), {len(data.get('edges', []))} edge(s)"
    ]
    if data.get("issues"):
        lines.append(f"Issues: {len(data['issues'])}")
    lines.append("Layers:")
    for layer in data.get("layers", [])[:8]:
        lines.append(f"- {layer.get('name')}: {len(layer.get('node_ids', []))} node(s)")
    lines.append("Tour:")
    for item in data.get("tour", [])[:8]:
        lines.append(f"- {item.get('order')}. {item.get('title')}: {len(item.get('node_ids', []))} node(s)")
    return "\n".join(lines)

## Definition order for the knowledge graph

The notebook executes from top to bottom. The catalog builder and its helpers therefore appear before the knowledge-graph builder, which calls `symbol_catalog`.

In [ ]:
#| export
def notebook_knowledge_graph_data(path="nbs"):
    """Return a typed graph of notebook, cell, symbol, import, and heading structure."""
    scope = _graph_scope(path)
    graph = _collect_graph(scope)
    catalog = symbol_catalog(scope)
    state = _kg_state()
    ordered_paths = [str(path) for path in _graph_limited_notebook_paths(scope)]
    for nb_order, nb_path in enumerate(ordered_paths):
        _kg_add_notebook(state, nb_path, nb_order, graph)
    _kg_add_call_edges(state, graph)
    _add_similarity_edges(catalog, state["nodes"], state["edges"], state["edge_seen"])
    data = dict(
        version=1, kind="nbskill_notebook_graph",
        project=_kg_project_summary(scope, ordered_paths, state),
        nodes=list(state["nodes"].values()), edges=state["edges"],
        layers=list(state["layers"].values()), tour=state["tour"], issues=[]
    )
    data["issues"] = _notebook_knowledge_graph_validate(data)
    return data

In [ ]:
#| export
def notebook_knowledge_graph(path="nbs", json_output=False):
    """Print a typed notebook knowledge graph summary, or JSON when requested."""
    data = notebook_knowledge_graph_data(path)
    if json_output:
        text = json.dumps(data, indent=2, sort_keys=True)
        if _GRAPH_JSON_MAX_CHARS and len(text) > _GRAPH_JSON_MAX_CHARS:
            raise ValueError(
                f"Graph JSON budget exceeded: {len(text)} chars exceeds limit {_GRAPH_JSON_MAX_CHARS}; "
                "narrow the path or use the summary output."
            )
        print(text)
        return cli_return(data)
    text = _format_notebook_knowledge_graph(data)
    print(text)
    return cli_return(text)

In [ ]:
#| hide
test_eq(isinstance(notebook_knowledge_graph(path="nbs/10_graph.ipynb"), str), True)

## 4. Offer conservative reuse advice

The advisor turns graph evidence into conservative suggestions. It never moves code; it records why an existing symbol or notebook is worth inspecting before an agent writes another helper.

In [ ]:
#| export
def _source_record_from_node(source, imports, symbol, symbol_node, kind):
    docstring = ast.get_docstring(symbol_node) or ""
    text = " ".join([source, symbol, kind, docstring, " ".join(imports)])
    return dict(
        symbol=symbol, kind=kind, module="", path="<source>", cell_id="", cell_idx=0,
        exported=False, private=symbol_short_name(symbol).startswith("_"),
        docstring=docstring, heading="", tokens=_tokens_from_text(text),
        calls=sorted(set(_call_names(symbol_node))), imports=imports,
        decorators=_decorator_names(symbol_node), ast_shape=_normalized_ast_shape(symbol_node)
    )

In [ ]:
#| export
def _source_advice_record(source):
    tree = ast.parse(source_without_directives(source))
    imports = _cell_import_names(tree)
    for node in tree.body:
        for symbol, symbol_node, kind in _node_definitions(node):
            return _source_record_from_node(source, imports, symbol, symbol_node, kind)
    return dict(
        symbol="", kind="source", path="<source>", cell_id="", cell_idx=0,
        exported=False, private=False, docstring="", heading="",
        tokens=_tokens_from_text(source), calls=[], imports=imports,
        decorators=[], ast_shape=""
    )

In [ ]:
#| export
def _advice_target(goal="", source=None, record=None):
    if record is not None: return dict(record)
    if source:
        try:
            target = _source_advice_record(source)
        except SyntaxError:
            target = _source_advice_record("pass")
            target["tokens"] = _tokens_from_text(source)
        target["tokens"] = sorted(set(target.get("tokens", [])) | set(_tokens_from_text(goal)))
        return target
    return dict(symbol="", tokens=_tokens_from_text(goal), calls=[], imports=[], ast_shape="")

In [ ]:
#| export
def _jaccard(left, right):
    left, right = set(left), set(right)
    if not left and not right: return 0.0
    return len(left & right) / max(1, len(left | right))

In [ ]:
#| export
def _reuse_score(target, record):
    name_score = 20 * _jaccard(_tokens_from_text(target.get("symbol", "")), record.get("tokens", []))
    token_score = 18 * _jaccard(target.get("tokens", []), record.get("tokens", []))
    call_score = 18 * _jaccard(
        map(symbol_short_name, target.get("calls", [])),
        map(symbol_short_name, record.get("calls", [])),
    )
    import_score = 10 * _jaccard(target.get("imports", []), record.get("imports", []))
    decorator_score = 8 * _jaccard(target.get("decorators", []), record.get("decorators", []))
    shape_score = 0
    if target.get("ast_shape") and target.get("ast_shape") == record.get("ast_shape"):
        shape_score = 45
    return round(name_score + token_score + call_score + import_score + decorator_score + shape_score, 2)

In [ ]:
#| export
def _reuse_reasons(target, record):
    reasons = []
    shared_tokens = set(target.get("tokens", [])) & set(record.get("tokens", []))
    shared_calls = set(map(symbol_short_name, target.get("calls", []))) & set(
        map(symbol_short_name, record.get("calls", []))
    )
    if shared_tokens: reasons.append("shared tokens: " + ", ".join(sorted(shared_tokens)[:6]))
    if shared_calls: reasons.append("shared calls: " + ", ".join(sorted(shared_calls)[:6]))
    if target.get("ast_shape") and target.get("ast_shape") == record.get("ast_shape"):
        reasons.append("normalized AST shape matches")
    if set(target.get("imports", [])) & set(record.get("imports", [])):
        reasons.append("shared imports")
    if set(target.get("decorators", [])) & set(record.get("decorators", [])):
        reasons.append("shared decorators")
    return reasons or ["name or notebook text overlap"]

In [ ]:
#| export
def _reuse_match(target, record, score):
    return dict(
        symbol=record["symbol"], kind=record["kind"], path=record["path"],
        cell_id=record["cell_id"], score=score,
        exported=record.get("exported", False), private=record.get("private", False),
        reasons=_reuse_reasons(target, record)
    )

In [ ]:
#| export
def _notebook_relevance(goal_tokens, notebook):
    token_score = 12 * _jaccard(goal_tokens, notebook.get("tokens", []))
    heading_tokens = _tokens_from_text(" ".join(notebook.get("headings", [])))
    heading_score = 10 * _jaccard(goal_tokens, heading_tokens)
    return round(token_score + heading_score, 2)

In [ ]:
#| export
def reuse_advice(goal, path="nbs", source=None, top_k=5):
    "Return existing symbols and notebooks to inspect before implementing code."
    catalog = symbol_catalog(path)
    target = _advice_target(goal=goal, source=source)
    matches = []
    for record in catalog["symbols"]:
        score = _reuse_score(target, record)
        if score <= 0: continue
        matches.append(_reuse_match(target, record, score))
    matches = sorted(matches, key=lambda item: (-item["score"], item["path"], item["symbol"]))[:top_k]
    notebooks = []
    goal_tokens = set(_tokens_from_text(goal)) | set(target.get("tokens", []))
    for notebook in catalog["notebooks"]:
        score = _notebook_relevance(goal_tokens, notebook)
        if score <= 0: continue
        notebooks.append({
            "path": notebook["path"],
            "module": notebook["module"],
            "score": score,
            "reasons": ["goal overlaps notebook headings, default_exp, imports, or symbols"],
        })
    notebooks = sorted(notebooks, key=lambda item: (-item["score"], item["path"]))[:top_k]
    return {"goal": goal, "matches": matches, "notebooks": notebooks, "top_k": top_k}

In [ ]:
#| export
def _caller_paths_for_symbol(graph, symbol):
    callers = []
    for caller in graph.get("callers", []):
        if any(_call_matches_symbol(call, symbol) for call in caller.get("calls", [])):
            callers.append(caller.get("path", ""))
    return callers

In [ ]:
#| export
def _score_notebook_for_target(target, notebook, graph):
    target_symbol = target.get("symbol", "")
    symbols = set(notebook.get("symbols", []))
    if notebook.get("path") == target.get("path"):
        symbols.discard(target_symbol)
    symbol_tokens = _tokens_from_text(" ".join(symbols))
    heading_tokens = _tokens_from_text(" ".join(notebook.get("headings", [])))
    import_tokens = _tokens_from_text(" ".join(notebook.get("imports", [])))
    calls = set(map(symbol_short_name, target.get("calls", [])))
    short_symbols = set(map(symbol_short_name, symbols))
    caller_paths = _caller_paths_for_symbol(graph, target_symbol) if target_symbol else []
    caller_count = caller_paths.count(notebook.get("path"))
    score = 0
    score += 14 * _jaccard(target.get("tokens", []), heading_tokens)
    score += 8 * _jaccard(target.get("tokens", []), symbol_tokens)
    score += 4 * _jaccard(target.get("imports", []), notebook.get("imports", []))
    score += 12 * _jaccard(calls, short_symbols)
    score += min(8, caller_count * 4)
    if target.get("private") and notebook.get("path") != target.get("path"): score -= 3
    if target.get("exported") and not symbols and notebook.get("path") == target.get("path"): score -= 2
    reasons = []
    if set(target.get("tokens", [])) & set(heading_tokens): reasons.append("heading text matches")
    if set(target.get("tokens", [])) & set(symbol_tokens): reasons.append("public symbols match")
    if calls & short_symbols: reasons.append("target calls symbols in this notebook")
    if caller_count: reasons.append(f"{caller_count} caller(s) already live here")
    if set(target.get("imports", [])) & set(notebook.get("imports", [])): reasons.append("imports match")
    return round(score, 2), reasons or ["weak notebook text match"]

In [ ]:
#| export
def _target_record(catalog, symbol=None, source=None):
    if symbol:
        for record in catalog["symbols"]:
            if record["symbol"] == symbol or symbol_short_name(record["symbol"]) == symbol:
                return dict(record)
    return _advice_target(source=source or "")

In [ ]:
#| export
def _placement_advice_from_catalog(catalog, target, notebook=None, top_k=5):
    target = dict(target)
    if notebook: target["path"] = str(notebook)
    candidates = []
    for nb_profile in catalog["notebooks"]:
        score, reasons = _score_notebook_for_target(target, nb_profile, catalog["graph"])
        candidates.append({
            "path": nb_profile["path"],
            "module": nb_profile["module"],
            "score": score,
            "reasons": reasons,
        })
    candidates = sorted(candidates, key=lambda item: (-item["score"], item["path"]))[:top_k]
    current = next((item for item in candidates if item["path"] == target.get("path")), None)
    best = candidates[0] if candidates else None
    current_score = current["score"] if current else 0
    gap = round((best["score"] if best else 0) - current_score, 2)
    return {
        "target": {
            "symbol": target.get("symbol", ""),
            "path": target.get("path", ""),
            "cell_id": target.get("cell_id", ""),
        },
        "current": current,
        "best": best,
        "gap": gap,
        "candidates": candidates,
    }

In [ ]:
#| export
def placement_advice(path="nbs", symbol=None, source=None, notebook=None, top_k=5):
    "Return ranked notebook candidates plus evidence for a symbol or source snippet."
    catalog = symbol_catalog(path)
    target = _target_record(catalog, symbol=symbol, source=source)
    return _placement_advice_from_catalog(catalog, target, notebook=notebook, top_k=top_k)

In [ ]:
#| export
def _advice_problem(code, record, detail="", **kwargs):
    return {
        "code": code,
        "path": record.get("path", ""),
        "cell_id": record.get("cell_id", ""),
        "detail": detail,
        "severity": "warning",
        "source": "nbskill",
        **kwargs,
    }

In [ ]:
#| export
def _similar_function_problems(catalog, threshold=55):
    problems = []
    for record in catalog["symbols"]:
        if not record.get("exported") or record.get("private"): continue
        best = None
        for other in catalog["symbols"]:
            if other is record or other.get("cell_id") == record.get("cell_id"): continue
            score = _reuse_score(record, other)
            if score < threshold: continue
            match = _reuse_match(record, other, score)
            if best is None or match["score"] > best["score"]: best = match
        if best is None: continue
        problems.append(_advice_problem(
            "similar-function",
            record,
            "inspect the existing symbol before keeping another helper",
            symbol=record["symbol"],
            candidate=best["symbol"],
            target_path=best["path"],
            target_cell_id=best["cell_id"],
            score=best["score"],
            confidence="high" if best["score"] >= 80 else "medium",
        ))
    return problems

In [ ]:
#| export
def _misplaced_function_problems(catalog, min_gap=1.5):
    problems = []
    for record in catalog["symbols"]:
        if not record.get("exported") or record.get("private"): continue
        advice = _placement_advice_from_catalog(catalog, record, top_k=5)
        best = advice.get("best") or {}
        if not best or best.get("path") == record.get("path"): continue
        if advice.get("gap", 0) < min_gap or best.get("score", 0) < 5: continue
        problems.append(_advice_problem(
            "misplaced-function",
            record,
            "another notebook is a stronger home for this symbol",
            symbol=record["symbol"],
            target_path=best.get("path"),
            score=best.get("score"),
            gap=advice.get("gap"),
            confidence="medium",
        ))
    return problems

In [ ]:
#| export
def _private_boundary_problems(catalog):
    graph = catalog["graph"]
    definitions = {(record.get("module"), record["symbol"]): record for record in graph["definitions"]}
    problems = []
    seen = set()
    for imported in graph["imports"]:
        symbol = imported["symbol"]
        if not symbol_short_name(symbol).startswith("_"): continue
        definition = definitions.get((imported["module"], symbol))
        if not definition or definition["path"] == imported["path"]: continue
        for caller in graph["callers"]:
            if caller["path"] != imported["path"]: continue
            if not any(_call_matches_symbol(call, imported["local"]) for call in caller["calls"]): continue
            key = (caller["path"], caller["cell_id"], symbol)
            if key in seen: continue
            seen.add(key)
            problems.append({
                "code": "private-boundary-reuse",
                "path": caller["path"],
                "cell_id": caller["cell_id"],
                "detail": "promote or move the helper instead of reusing it across notebooks privately",
                "severity": "warning",
                "source": "nbskill",
                "symbol": imported["local"],
                "candidate": symbol,
                "target_path": definition["path"],
                "target_cell_id": definition["cell_id"],
                "confidence": "high",
            })
    return problems

In [ ]:
#| export
def notebook_advice_problems(path="nbs"):
    "Return reuse and placement advisory diagnostics for notebook style reports."
    catalog = symbol_catalog(path)
    problems = []
    problems.extend(_similar_function_problems(catalog))
    problems.extend(_misplaced_function_problems(catalog))
    problems.extend(_private_boundary_problems(catalog))
    return problems

In [ ]:
#| hide
root = demo_path("10_taste_advisor")
remove_demo_path(root)
root.mkdir(parents=True)
write_nb(new_nb([
    mk_cell("#| default_exp parser"),
    mk_cell("## Parser helpers", cell_type="markdown"),
    mk_cell("#| export\ndef parse_value(text):\n    return text.strip().lower()"),
]), root / "00_parse.ipynb")

In [ ]:
#| hide
write_nb(new_nb([
    mk_cell("#| default_exp misc"),
    mk_cell("## Misc utilities", cell_type="markdown"),
    mk_cell("#| export\ndef clean_value(value):\n    return value.strip().lower()"),
    mk_cell("#| export\ndef parse_tokens(text):\n    return parse_value(text).split()"),
]), root / "01_misc.ipynb")

In [ ]:
#| hide
write_nb(new_nb([
    mk_cell("#| default_exp alpha"),
    mk_cell("#| export\ndef _shared():\n    return 1"),
    mk_cell("#| export\ndef _local():\n    return 2\n\ndef public_local():\n    return _local()"),
]), root / "02_alpha.ipynb")

In [ ]:
#| hide
write_nb(new_nb([
    mk_cell("#| default_exp beta"),
    mk_cell("#| export\nfrom alpha import _shared\n\ndef public_value():\n    return _shared()"),
]), root / "03_beta.ipynb")

In [ ]:
#| hide
catalog = symbol_catalog(str(root))
symbols = {record["symbol"] for record in catalog["symbols"]}
assert {"parse_value", "clean_value", "parse_tokens", "_shared"} <= symbols
assert any(record["heading"] == "Parser helpers" for record in catalog["symbols"])

reuse = reuse_advice(
    "normalize parser text values",
    path=str(root),
    source="def tidy_text(raw):\n    return raw.strip().lower()",
)
assert reuse["matches"]
assert reuse["matches"][0]["symbol"] in {"parse_value", "clean_value"}

In [ ]:
#| hide
placement = placement_advice(path=str(root), symbol="parse_tokens", top_k=4)
assert placement["best"]["path"].endswith("00_parse.ipynb")
assert placement["gap"] > 0

problems = notebook_advice_problems(str(root))
assert any(problem["code"] == "similar-function" for problem in problems)
assert any(
    problem["code"] == "misplaced-function" and problem.get("symbol") == "parse_tokens"
    for problem in problems
)
assert any(
    problem["code"] == "private-boundary-reuse" and problem.get("candidate") == "_shared"
    for problem in problems
)
assert not [
    problem for problem in problems
    if problem["code"] == "private-boundary-reuse" and problem.get("candidate") == "_local"
]

In [ ]:
#| hide
remove_demo_path(root)

In [ ]:
#| hide
root = demo_path("10_graph_connection_tests")
try:
    root.mkdir()
    lib = root / "lib.ipynb"
    write_nb(new_nb([
        mk_cell("#| default_exp lib"),
        mk_cell("#| export\ndef leaf():\n    return 1\n\ndef middle():\n    return leaf()\n\ndef start():\n    return middle()"),
    ]), lib)
    data = _symbol_connection_data(str(root), "start", "leaf")
    assert data["found"]
    assert data["symbols"] == ["start", "middle", "leaf"]
    assert [edge["call"] for edge in data["chain"]] == ["middle", "leaf"]
    text = capture_call(symbol_connection, path=str(root), start="start", end="leaf")
    assert "Symbols: start -> middle -> leaf" in text
    assert "calls middle" in text and "calls leaf" in text
    json_text = capture_call(symbol_connection, path=str(root), start="start", end="leaf", json_output=True)
    assert '"symbols"' in json_text and '"chain"' in json_text
    missing = _symbol_connection_data(str(root), "leaf", "start")
    assert not missing["found"]
    print("symbol_connection chain:", " -> ".join(data["symbols"]))
finally:
    remove_demo_path(root)

In [ ]:
def _write_graph_test_notebooks(root):
    root.mkdir()
    lib = root / "lib.ipynb"
    caller = root / "caller.ipynb"
    write_nb(new_nb([
        mk_cell("#| default_exp lib"),
        mk_cell("#| export\ndef helper():\n    return 1\n\ndef _secret():\n    return helper()\n\ndef target():\n    return helper()"),
    ]), lib)
    write_nb(new_nb([mk_cell("from nbskill.lib import target, _secret\nvalue = target()\n_secret()")]), caller)
    return lib, caller

In [ ]:
#| hide
def _write_knowledge_graph_test_notebooks(root):
    root.mkdir()
    lib = root / "kg.ipynb"
    caller = root / "caller.ipynb"
    write_nb(new_nb([
        mk_cell("#| default_exp kg", cell_type="code"),
        mk_cell("## Core helpers\nTiny demo section.", cell_type="markdown"),
        mk_cell(
            "#| export\n"
            "def helper():\n"
            "    return 1\n\n"
            "def clone():\n"
            "    return 1\n\n"
            "def target():\n"
            "    return helper()",
            cell_type="code",
        ),
        mk_cell("target()", cell_type="code"),
    ]), lib)
    write_nb(new_nb([
        mk_cell("from nbskill.kg import target\nvalue = target()", cell_type="code"),
    ]), caller)
    return lib, caller

In [ ]:
#| hide
root = demo_path("10_notebook_knowledge_graph")
_write_knowledge_graph_test_notebooks(root)
data = notebook_knowledge_graph_data(str(root))

In [ ]:
#| hide
node_types = {node["type"] for node in data["nodes"]}
edge_types = {edge["type"] for edge in data["edges"]}
assert {"notebook", "cell", "symbol", "heading", "import"} <= node_types
assert {"contains", "defines", "calls", "imports", "documents", "precedes", "similar_to"} <= edge_types
assert data["kind"] == "nbskill_notebook_graph"
assert data["issues"] == []
node_ids = {node["id"] for node in data["nodes"]}
for layer in data["layers"]:
    assert set(layer["node_ids"]) <= node_ids
for item in data["tour"]:
    assert set(item["node_ids"]) <= node_ids

In [ ]:
#| hide
bad = dict(data)
bad.update(
    nodes=[*data["nodes"], data["nodes"][0]],
    edges=[
        *data["edges"],
        dict(source="missing", target=data["nodes"][0]["id"], type="calls", confidence="low", evidence="bad"),
    ],
    layers=[dict(id="bad-layer", name="Bad", node_ids=["missing"])],
    tour=[dict(order=0, title="Bad", node_ids=["missing"])],
    issues=[],
)
codes = {issue["code"] for issue in _notebook_knowledge_graph_validate(bad)}
assert {"duplicate-node-id", "dangling-edge", "dangling-layer-node", "dangling-tour-node"} <= codes

In [ ]:
#| hide
json_text = capture_call(notebook_knowledge_graph, path=str(root), json_output=True)
parsed = json.loads(json_text)
assert parsed["kind"] == "nbskill_notebook_graph"
assert parsed == notebook_knowledge_graph_data(str(root))
print(f"notebook knowledge graph: {len(data['nodes'])} nodes, {len(data['edges'])} edges")

In [ ]:
#| hide
remove_demo_path(root)

In [ ]:
#| hide
root = demo_path("10_graph_symbol")
try:
    _write_graph_test_notebooks(root)
    text = capture_call(symbol_graph, path=str(root), symbol="target")
    assert "Definitions:" in text
    assert "caller.ipynb id=" in text
    assert "Caller usages:" in text
    assert "value = target()" in text
    assert "helper" in text
    json_text = capture_call(symbol_graph, path=str(root), symbol="target", json_output=True)
    structured = symbol_graph_public_data(symbol_graph_data(str(root), "target"))
    assert '"caller_usages"' in json_text
    assert len(structured["caller_usages"]) == 1
    assert structured["caller_usages"][0]["line"] == "value = target()"
    print(f"structured symbol_graph usages: {len(structured['caller_usages'])}")
    summary = symbol_usage_summary(str(root), ["target"])
    assert "callers=" in summary and "caller.ipynb id=" in summary
    assert "Caller usages:" in summary and "value = target()" in summary
finally:
    remove_demo_path(root)

In [ ]:
#| hide
root = demo_path("10_graph_order")
try:
    root.mkdir()
    order_nb = root / "order.ipynb"
    write_nb(new_nb([
        mk_cell("result = later_helper()"),
        mk_cell("def later_helper():\n    return 1"),
        mk_cell("def loader():\n    return MissingPath('x')"),
        mk_cell("with open(__file__) as handle:\n    out = handle.read()\ntext = out.strip()"),
        mk_cell("from pathlib import Path\npath = Path('x')\nvalue = (path / 'y').exists()\njoined = 'a'.join(['b'])"),
    ]), order_nb)
    order_lines = notebook_order_problem_lines(str(order_nb))
    assert any("cell-order" in line and "later_helper" in line for line in order_lines)
    assert any("missing-import" in line and "MissingPath" in line for line in order_lines)
    assert not any("symbol='exists'" in line or "symbol='join'" in line or "symbol='out'" in line for line in order_lines)
    print("with-block locals are ignored by order warnings")
finally:
    remove_demo_path(root)

In [ ]:
#| hide
root = demo_path("10_graph_private")
try:
    _write_graph_test_notebooks(root)
    report = capture_call(private_symbol_report, path=str(root))
    assert "_secret" in report
    assert "called from" in report
finally:
    remove_demo_path(root)

## 5. Explore the graph

The symbol graph can answer questions that plain text search only approximates. It can connect a definition to the cells that call it, show what that definition itself calls, rank symbols by likely edit blast radius, and expose private helpers that have crossed notebook boundaries.

These prototype cells stay non-exported on purpose. They test strategies over the current `nbs` graph before deciding whether any report deserves a public CLI or MCP surface. The results are static-analysis hints, not deletion permission: dynamic imports, CLI entrypoints, examples, and short-name collisions still need human review.

In [ ]:
#| hide
from collections import Counter, defaultdict

In [ ]:
#| hide
from nbskill.review import style_report

In [ ]:
def _explore_definition_id(record): return (record.get("module"), record["symbol"], record["path"], record["cell_id"])

In [ ]:
def _explore_call_sites_by_name(graph):
    sites = defaultdict(list)
    for caller in graph["callers"]:
        for site in caller.get("call_sites", ()):
            sites[site.get("name")].append(dict(path=caller["path"], cell_id=caller["cell_id"], cell_idx=caller["cell_idx"], lineno=site.get("lineno"), line=site.get("line")))
    return sites

In [ ]:
def _explore_unique_sites(rows):
    seen, unique = set(), []
    for row in rows:
        key = (row["path"], row["cell_id"], row.get("lineno"), row.get("line"))
        if key in seen: continue
        seen.add(key)
        unique.append(row)
    return unique

In [ ]:
def _explore_callers_by_definition(graph):
    sites = _explore_call_sites_by_name(graph)
    callers = {}
    for record in graph["definitions"]:
        names = dict.fromkeys([record["symbol"], symbol_short_name(record["symbol"])])
        rows = [site for name in names for site in sites.get(name, ())]
        callers[_explore_definition_id(record)] = _explore_unique_sites(rows)
    return callers

In [ ]:
def _explore_unused_candidates(graph):
    callers = _explore_callers_by_definition(graph)
    result = {"public": [], "private": []}
    for record in graph["definitions"]:
        if callers[_explore_definition_id(record)]: continue
        visibility = "private" if record["symbol"].startswith("_") else "public"
        result[visibility].append(dict(symbol=record["symbol"], kind=record["kind"], module=record.get("module"), path=record["path"], cell_id=record["cell_id"], caveat="check CLI, docs, and dynamic calls before deleting"))
    for rows in result.values(): rows.sort(key=lambda row: (row["module"] or "", row["symbol"]))
    return result

In [ ]:
def _explore_hotspots(graph):
    callers = _explore_callers_by_definition(graph)
    rows = []
    for record in graph["definitions"]:
        fan_in = len({(site["path"], site["cell_id"]) for site in callers[_explore_definition_id(record)]})
        fan_out = len([symbol for symbol in _resolve_callees(graph, record.get("calls", ())) if symbol != record["symbol"]])
        if fan_in or fan_out:
            rows.append(dict(symbol=record["symbol"], module=record.get("module"), path=record["path"], cell_id=record["cell_id"], fan_in=fan_in, fan_out=fan_out, score=fan_in * 2 + fan_out))
    return sorted(rows, key=lambda row: (row["score"], row["fan_in"], row["fan_out"]), reverse=True)

In [ ]:
def _explore_private_boundary_leaks(graph):
    definitions = {(row.get("module"), row["symbol"]): row for row in graph["definitions"]}
    rows = []
    for imported in graph["imports"]:
        if not symbol_short_name(imported["symbol"]).startswith("_"): continue
        definition = definitions.get((imported["module"], imported["symbol"]))
        if not definition or definition["path"] == imported["path"]: continue
        for caller in graph["callers"]:
            if caller["path"] != imported["path"]: continue
            sites = [site for site in caller.get("call_sites", ()) if _call_matches_symbol(site.get("name"), imported["local"])]
            if sites:
                row = dict(symbol=imported["symbol"], local=imported["local"], defined_path=definition["path"])
                row.update(defined_cell=definition["cell_id"], called_path=caller["path"], called_cell=caller["cell_id"], line=sites[0].get("line"))
                rows.append(row)
    return rows

In [ ]:
def _explore_ambiguous_short_names(graph):
    sites = _explore_call_sites_by_name(graph)
    by_short = defaultdict(list)
    for record in graph["definitions"]: by_short[symbol_short_name(record["symbol"])].append(record)
    rows = []
    for name, records in by_short.items():
        locations = {(row.get("module"), row["symbol"], row["path"], row["cell_id"]) for row in records}
        if len(locations) <= 1: continue
        row = dict(short_name=name, definition_count=len(locations), call_site_count=len(sites.get(name, ())))
        row.update(symbols=sorted({row["symbol"] for row in records})[:6], caveat="short-name matching can overconnect this symbol")
        rows.append(row)
    return sorted(rows, key=lambda row: (row["call_site_count"], row["definition_count"]), reverse=True)

In [ ]:
def _explore_notebook_hotspots(graph, hotspots, private_leaks, order_problems=(), style=None):
    def_counts = Counter(row["path"] for row in graph["definitions"])
    central = Counter(row["path"] for row in hotspots[:30])
    leaks = Counter(row["called_path"] for row in private_leaks)
    order = Counter(row["path"] for row in order_problems)
    style_rows = (style or {}).get("notebook_problems", ())
    style_counts = Counter(row["path"] for row in style_rows)
    rows = []
    for path in set(def_counts) | set(central) | set(leaks) | set(order) | set(style_counts):
        score = central[path] * 3 + leaks[path] * 2 + order[path] + style_counts[path]
        row = dict(path=path, definitions=def_counts[path], central_symbols=central[path])
        row.update(private_leaks=leaks[path], order_warnings=order[path], style_warnings=style_counts[path], score=score)
        rows.append(row)
    return sorted(rows, key=lambda row: (row["score"], row["definitions"]), reverse=True)

The first useful reports are comparative rather than absolute. A symbol with no resolved callers is a cleanup candidate, not proof of dead code. A high fan-in or fan-out symbol deserves more careful tests before editing. A private boundary leak shows where notebook-local code has become a cross-module dependency. Ambiguous short names show where the current graph can lie by matching `Class.method` and `method` too generously.

In [ ]:
_explore_scope = "nbs/*.ipynb"
_explore_notebook = notebook_paths("nbs/10_graph.ipynb")[0]
_explore_graph = _collect_graph(_explore_scope)
_explore_style = style_report(_explore_notebook)
_explore_unused = _explore_unused_candidates(_explore_graph)
_explore_hotspot_rows = _explore_hotspots(_explore_graph)
_explore_private_leaks = _explore_private_boundary_leaks(_explore_graph)
_explore_ambiguous = _explore_ambiguous_short_names(_explore_graph)
_explore_notebooks = _explore_notebook_hotspots(_explore_graph, _explore_hotspot_rows, _explore_private_leaks, notebook_order_problems(_explore_scope), _explore_style)
_explore_connection_pairs = [
    ("symbol_connection", "_resolve_callees"),
    ("symbol_graph", "_caller_usage_records"),
    ("style_check", "notebook_order_problem_lines"),
]
_explore_connections = [
    _symbol_connection_public_data(_symbol_connection_data(_explore_scope, start, end, max_depth=6))
    for start, end in _explore_connection_pairs
]

print(f"graph: {len(_explore_graph['definitions'])} definitions, {len(_explore_graph['callers'])} caller cells, {len(_explore_graph['imports'])} imports")
print("hotspots:", ", ".join(row["symbol"] for row in _explore_hotspot_rows[:8]))
print("private leaks:", ", ".join(row["symbol"] for row in _explore_private_leaks[:6]))
print("unused public candidates:", ", ".join(row["symbol"] for row in _explore_unused["public"][:6]))
print("ambiguous names:", ", ".join(row["short_name"] for row in _explore_ambiguous[:6]))
print("notebook hotspots:", ", ".join(row["path"] for row in _explore_notebooks[:4]))
for row in _explore_connections:
    chain = " -> ".join(row["symbols"]) if row["found"] else "not found"
    print(f"connection {row['start']} to {row['end']}: {chain}")

In [ ]:
#| hide
assert all({"symbol", "fan_in", "fan_out", "score"} <= set(row) for row in _explore_hotspot_rows[:5])
assert all({"symbol", "defined_path", "called_path", "line"} <= set(row) for row in _explore_private_leaks[:5])
assert all({"short_name", "definition_count", "call_site_count", "symbols"} <= set(row) for row in _explore_ambiguous[:5])
assert _explore_notebooks and all({"path", "score", "definitions"} <= set(row) for row in _explore_notebooks[:5])
assert not any(row["short_name"] in dir(builtins) for row in _explore_ambiguous[:20])
assert len(_explore_connections) == len(_explore_connection_pairs)
assert all({"start", "end", "found", "symbols"} <= set(row) for row in _explore_connections)
print("graph exploration prototypes produced structured, seeded findings")

#### Public behavior

The order checker exposes calls that appear before their definitions or imports, which makes notebook execution problems visible before runtime.

In [ ]:
_order_examples = notebook_order_problems("nbs/10_graph.ipynb")
print(f"order diagnostics: {len(_order_examples)}")

In [ ]:
#| hide
assert isinstance(_order_examples, list)

#### notebook_order_problem_lines

`notebook_order_problem_lines` converts the structured warnings into concise style-check lines that are easy to print or filter.

In [ ]:
_order_lines_example = notebook_order_problem_lines("nbs/10_graph.ipynb")
print("\n".join(_order_lines_example[:3]) or "no order warnings")

In [ ]:
#| hide
assert isinstance(_order_lines_example, list)

#### symbol_graph_data

`symbol_graph_data` is the structured entry point for callers that need definitions, caller sites, and callee information without parsing report text.

In [ ]:
_graph_data_example = symbol_graph_data("nbs/10_graph.ipynb", "symbol_graph")
print(dict(symbol=_graph_data_example["symbol"], definitions=len(_graph_data_example["definitions"]), callers=len(_graph_data_example["callers"])))

In [ ]:
#| hide
assert _graph_data_example["symbol"] == "symbol_graph"
assert {"definitions", "callers", "callees"} <= set(_graph_data_example)

#### symbol_graph_public_data

`symbol_graph_public_data` keeps the automation-facing shape small while preserving caller usage records.

In [ ]:
_public_graph_example = symbol_graph_public_data(_graph_data_example)
print(dict(symbol=_public_graph_example["symbol"], callers=len(_public_graph_example["callers"]), usages=len(_public_graph_example["caller_usages"])))

In [ ]:
#| hide
assert _public_graph_example["symbol"] == "symbol_graph"
assert "caller_usages" in _public_graph_example

#### symbol_usage_summary

`symbol_usage_summary` is the compact human-facing view for checking several symbols at once.

In [ ]:
_usage_summary_example = symbol_usage_summary("nbs/10_graph.ipynb", ["symbol_graph", "symbol_connection"])
print(_usage_summary_example.splitlines()[0])

In [ ]:
#| hide
assert "symbol_graph" in _usage_summary_example

#### Public behavior

The report function prints a short orientation view for a symbol; use the data functions above when a caller needs machine-readable details.

In [ ]:
symbol_graph(path="nbs/10_graph.ipynb", symbol="symbol_graph")

In [ ]:
#| hide
assert _graph_data_example["definitions"]

#### private_symbol_report

`private_symbol_report` is a boundary check for private helpers imported across notebooks.

In [ ]:
private_symbol_report(path="nbs/10_graph.ipynb")

In [ ]:
#| hide
assert isinstance(private_symbol_report(path="nbs/10_graph.ipynb"), (str, type(None)))

#### symbol_connection

`symbol_connection` follows the shortest approximate call chain between two symbols and prints whether a connection was found.

In [ ]:
symbol_connection(path="nbs/10_graph.ipynb", start="symbol_graph", end="_collect_graph")

In [ ]:
#| hide
assert _symbol_connection_data("nbs/10_graph.ipynb", "symbol_graph", "_collect_graph")["start"] == "symbol_graph"

#### notebook_knowledge_graph_data

`notebook_knowledge_graph_data` returns a typed graph of notebook, cell, symbol, import, and heading nodes.

In [ ]:
_knowledge_graph_example = notebook_knowledge_graph_data("nbs/10_graph.ipynb")
print(dict(nodes=len(_knowledge_graph_example["nodes"]), edges=len(_knowledge_graph_example["edges"]), issues=len(_knowledge_graph_example["issues"])))

In [ ]:
#| hide
assert _knowledge_graph_example["kind"] == "nbskill_notebook_graph"
assert _knowledge_graph_example["issues"] == []

#### notebook_knowledge_graph

`notebook_knowledge_graph` renders the typed graph as a concise summary for quick inspection.

In [ ]:
notebook_knowledge_graph(path="nbs/10_graph.ipynb")

In [ ]:
#| hide
assert _knowledge_graph_example["nodes"]

#### symbol_catalog

`symbol_catalog` records where symbols live and the text evidence that helps agents decide what to reuse.

In [ ]:
_catalog_example = symbol_catalog("nbs/10_graph.ipynb")
print(f"catalog symbols: {len(_catalog_example['symbols'])}")

In [ ]:
#| hide
assert _catalog_example["symbols"]

#### reuse_advice

`reuse_advice` ranks existing symbols and notebooks before new code is written.

In [ ]:
_reuse_example = reuse_advice("notebook graph parser", path="nbs/10_graph.ipynb", top_k=3)
print(f"reuse matches: {len(_reuse_example['matches'])}")

In [ ]:
#| hide
assert isinstance(_reuse_example, dict)
assert "matches" in _reuse_example

#### placement_advice

`placement_advice` ranks likely notebook locations for a symbol or source snippet.

In [ ]:
_placement_example = placement_advice(path="nbs/10_graph.ipynb", symbol="symbol_graph", top_k=3)
print(f"placement candidates: {len(_placement_example['candidates'])}")

In [ ]:
#| hide
assert _placement_example["target"]["symbol"] == "symbol_graph"

#### notebook_advice_problems

`notebook_advice_problems` collects conservative reuse, placement, and private-boundary diagnostics for style reports.

In [ ]:
_advice_problems_example = notebook_advice_problems("nbs/10_graph.ipynb")
print(f"advice diagnostics: {len(_advice_problems_example)}")

In [ ]:
#| hide
assert isinstance(_advice_problems_example, list)